# Generation MIDI avec midi-model (SkyTNT)

**Module :** 02-Audio-Advanced  
**Niveau :** Intermediaire  
**Technologies :** midi-model (SkyTNT), FluidSynth, ~2-4 GB VRAM  
**Duree estimee :** 45 minutes  

## Objectifs d'Apprentissage

- [ ] Comprendre la difference entre generation symbolique (MIDI) et generation audio directe
- [ ] Installer et charger le modele midi-model (architecture dual-transformer)
- [ ] Generer des sequences MIDI multi-instruments de maniere inconditionnelle
- [ ] Configurer des prompts musicaux (tempo, instruments, tonalite)
- [ ] Maitriser les parametres de sampling (temperature, top_k, top_p)
- [ ] Synthetiser le MIDI en audio avec FluidSynth
- [ ] Comparer generation symbolique (MIDI) vs generation audio directe (MusicGen)

## Prerequis

- GPU NVIDIA avec au moins 2 GB VRAM (ou CPU, plus lent)
- FluidSynth >= 2.0 installe au niveau systeme
- `pip install torch safetensors peft transformers pyfluidsynth tqdm huggingface_hub`
- Connaissances de base en theorie musicale (gammes, accords) utiles mais pas indispensables

**Navigation :** [<< 02-5](02-5-Multi-Model-TTS-Gateway.ipynb) | [Index](../README.md) | [03-1 >>](../03-Orchestration/03-1-Multi-Model-Audio-Comparison.ipynb)

In [ ]:
# Parametres Papermill - JAMAIS modifier ce commentaire

# Configuration notebook
notebook_mode = "interactive"        # "interactive" ou "batch"
skip_widgets = False               # True pour mode batch MCP
debug_level = "INFO"

# Parametres midi-model
model_id = "skytnt/midi-model-tv2o-medium"  # HuggingFace model ID
config_name = "tv2o-medium"                  # Config name
max_len = 512                                # Nombre max d'evenements MIDI
device = "cuda"                              # "cuda" ou "cpu"

# Parametres de sampling
temperature = 1.0                  # Temperature (0.5-1.2)
top_k = 20                         # Top-K sampling
top_p = 0.98                       # Nucleus sampling

# Configuration
generate_midi = True               # Generer les fichiers MIDI
save_results = True                # Sauvegarder les fichiers
synthesize_audio = True            # Synthetiser MIDI en audio
test_prompts = True                # Tester les prompts musicaux

Les parametres Papermill definissent le modele midi-model (`skytnt/midi-model-tv2o-medium`), les parametres de sampling (temperature, top_k, top_p) et les options de generation. La cellule suivante initialise l'environnement Python et verifie la disponibilite du GPU.

In [2]:
# Setup environnement et imports
import os
import sys
import time
import gc
import tempfile
import subprocess
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional
import logging

import numpy as np
from IPython.display import Audio, display, HTML

# Import helpers GenAI
GENAI_ROOT = Path.cwd()
while GENAI_ROOT.name != 'GenAI' and len(GENAI_ROOT.parts) > 1:
    GENAI_ROOT = GENAI_ROOT.parent

HELPERS_PATH = GENAI_ROOT / 'shared' / 'helpers'
if HELPERS_PATH.exists():
    sys.path.insert(0, str(HELPERS_PATH.parent))
    try:
        from helpers.audio_helpers import play_audio, save_audio
        print("Helpers audio importes")
    except ImportError:
        print("Helpers audio non disponibles - mode autonome")

# Repertoires
OUTPUT_DIR = GENAI_ROOT / 'outputs' / 'audio' / 'midi'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Repertoire pour le code source midi-model
MIDI_MODEL_DIR = GENAI_ROOT / 'models' / 'midi-model'
MIDI_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Configuration logging
logging.basicConfig(level=getattr(logging, debug_level))
logger = logging.getLogger('midi-generation')

# Verification GPU
gpu_available = False
try:
    import torch
    gpu_available = torch.cuda.is_available()
    if gpu_available:
        gpu_name = torch.cuda.get_device_name(0)
        gpu_vram = torch.cuda.get_device_properties(0).total_mem / (1024**3)
        print(f"GPU : {gpu_name} ({gpu_vram:.1f} GB VRAM)")
    else:
        print("GPU non disponible - generation plus lente sur CPU")
        if device == "cuda":
            device = "cpu"
            print("Fallback vers CPU")
except ImportError:
    print("torch non installe")
    device = "cpu"

print(f"\nGeneration MIDI avec midi-model")
print(f"Date : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Mode : {notebook_mode}, Device : {device}")
print(f"Modele : {model_id}")
print(f"Max events : {max_len}, Temperature : {temperature}")
print(f"Sortie : {OUTPUT_DIR}")

Helpers audio importes


GPU non disponible - generation plus lente sur CPU
Fallback vers CPU

Generation MIDI avec midi-model
Date : 2026-03-22 13:33:50
Mode : interactive, Device : cpu
Modele : skytnt/midi-model-tv2o-medium
Max events : 512, Temperature : 1.0
Sortie : D:\Dev\CoursIA.worktrees\GenAI_Series\MyIA.AI.Notebooks\GenAI\outputs\audio\midi


L'environnement Python est initialise et les repertoires de sortie sont crees. La cellule suivante charge le fichier `.env` pour recuperer le token HuggingFace, utile pour le telechargement des poids de `skytnt/midi-model`.

In [3]:
# Chargement robuste de la configuration .env
from dotenv import load_dotenv
import os

current_path = Path.cwd()
env_loaded = False
for _ in range(10):
    env_path = current_path / ".env"
    if env_path.exists():
        load_dotenv(env_path)
        print(f".env charge depuis: {env_path}")
        env_loaded = True
        break
    current_path = current_path.parent
    if current_path.name == "GenAI" or len(current_path.parts) <= 1:
        break
if not env_loaded:
    print("WARNING: .env non trouve, utilisation variables environnement")

# HuggingFace token (optionnel, modele public)
hf_token = os.environ.get("HUGGINGFACE_TOKEN") or os.environ.get("HF_TOKEN")
if hf_token:
    print("Token HuggingFace disponible")
else:
    print("Token HuggingFace non disponible (telechargement public uniquement)")

Token HuggingFace non disponible (telechargement public uniquement)


## Dependances GPU Optionnelles
Ce notebook utilise un modele Transformer leger (~200 MB). GPU recommande pour la vitesse, mais le CPU fonctionne.
FluidSynth est requis au niveau systeme pour la synthese audio des fichiers MIDI.

## Section 1 : Generation symbolique vs generation audio

Il existe deux approches fondamentalement differentes pour generer de la musique par IA :

| Aspect | Generation symbolique (MIDI) | Generation audio (MusicGen) |
|--------|------------------------------|-----------------------------|
| Sortie | Notes, durees, instruments | Signal audio brut |
| Editabilite | Totale (notes individuelles) | Aucune (forme d'onde) |
| Taille | Quelques KB (.mid) | Plusieurs MB (.wav) |
| Multi-instruments | Natif (16 canaux MIDI) | Melange dans le signal |
| Qualite sonore | Depend du synthetiseur | Directement realiste |
| VRAM | ~2-4 GB | ~10-20 GB |

### Architecture de midi-model

midi-model utilise une architecture **dual-transformer** basee sur LLaMA :

| Composant | Role | Taille (medium) |
|-----------|------|------------------|
| **net** (Main) | Traite la sequence d'evenements MIDI | 12 couches, 16 tetes, 1024 hidden |
| **net_token** (Token) | Genere les tokens individuels par evenement | 3 couches, 4 tetes, 1024 hidden |
| **Tokenizer V2** | Encode/decode les evenements MIDI | Vocabulaire : 3406 tokens |

Le tokenizer V2 represente chaque evenement MIDI par jusqu'a 8 tokens : type, temps, piste, canal, pitch, velocity, duree.

In [4]:
# Installation et telechargement du modele
print("INSTALLATION ET CHARGEMENT")
print("=" * 45)

# Verifier les dependances
missing_deps = []
for pkg_name, import_name in [("safetensors", "safetensors"), ("peft", "peft"),
                               ("transformers", "transformers"), ("tqdm", "tqdm"),
                               ("huggingface_hub", "huggingface_hub")]:
    try:
        __import__(import_name)
    except ImportError:
        missing_deps.append(pkg_name)

if missing_deps:
    print(f"Dependances manquantes : {', '.join(missing_deps)}")
    print(f"Installation : pip install {' '.join(missing_deps)}")
else:
    print("Toutes les dependances Python sont installees")

# Verifier FluidSynth
fluidsynth_available = False
try:
    import fluidsynth
    fluidsynth_available = True
    print("pyfluidsynth disponible")
except ImportError:
    print("pyfluidsynth non installe (pip install pyfluidsynth)")
    print("FluidSynth systeme requis egalement")

# Telecharger les fichiers source du modele depuis HuggingFace
from huggingface_hub import hf_hub_download, snapshot_download

source_files = ["midi_model.py", "midi_tokenizer.py", "midi_synthesizer.py", "MIDI.py"]
model_files_ready = True

print(f"\nTelechargement des sources midi-model...")
for fname in source_files:
    target = MIDI_MODEL_DIR / fname
    if not target.exists():
        try:
            downloaded = hf_hub_download(
                repo_id="skytnt/midi-model-tv2o-medium",
                filename=fname,
                local_dir=str(MIDI_MODEL_DIR),
                token=hf_token
            )
            print(f"  Telecharge : {fname}")
        except Exception as e:
            print(f"  Erreur {fname} : {str(e)[:80]}")
            model_files_ready = False
    else:
        print(f"  Deja present : {fname}")

# Telecharger le soundfont pour la synthese
soundfont_path = None
if synthesize_audio:
    sf_target = MIDI_MODEL_DIR / "soundfont.sf2"
    if not sf_target.exists():
        try:
            soundfont_path = hf_hub_download(
                repo_id="skytnt/midi-model",
                filename="soundfont.sf2",
                local_dir=str(MIDI_MODEL_DIR),
                token=hf_token
            )
            print(f"  Telecharge : soundfont.sf2")
        except Exception as e:
            print(f"  Erreur soundfont : {str(e)[:80]}")
    else:
        soundfont_path = str(sf_target)
        print(f"  Deja present : soundfont.sf2")

if model_files_ready:
    print(f"\nFichiers source prets dans : {MIDI_MODEL_DIR}")

INSTALLATION ET CHARGEMENT


Dependances manquantes : peft
Installation : pip install peft
pyfluidsynth non installe (pip install pyfluidsynth)
FluidSynth systeme requis egalement

Telechargement des sources midi-model...


INFO:httpx:HTTP Request: HEAD https://huggingface.co/skytnt/midi-model-tv2o-medium/resolve/main/midi_model.py "HTTP/1.1 404 Not Found"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/skytnt/midi-model-tv2o-medium/resolve/main/midi_tokenizer.py "HTTP/1.1 404 Not Found"


  Erreur midi_model.py : 404 Client Error. (Request ID: Root=1-69bfe1d6-4f34be6b12613b722495c99f;81bba1b7
  Erreur midi_tokenizer.py : 404 Client Error. (Request ID: Root=1-69bfe1d6-4563bc844e89fc6a4e975200;e8156c45


INFO:httpx:HTTP Request: HEAD https://huggingface.co/skytnt/midi-model-tv2o-medium/resolve/main/midi_synthesizer.py "HTTP/1.1 404 Not Found"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/skytnt/midi-model-tv2o-medium/resolve/main/MIDI.py "HTTP/1.1 404 Not Found"


  Erreur midi_synthesizer.py : 404 Client Error. (Request ID: Root=1-69bfe1d6-4f682c44149f0068256ad9eb;0eb4d4e9
  Erreur MIDI.py : 404 Client Error. (Request ID: Root=1-69bfe1d6-62e5bac80f6498ec7cbc259e;f05e831e
  Deja present : soundfont.sf2


Les dependances Python (`peft`, `pyfluidsynth`) et les fichiers sources du depot `midi-model` sont telecharges. La cellule suivante charge le modele en memoire et initialise le tokenizer V2 (vocabulaire de 3406 tokens).

In [5]:
# Chargement du modele midi-model
print("CHARGEMENT DU MODELE")
print("=" * 45)

midi_model_loaded = False

if model_files_ready:
    # Ajouter le repertoire au path
    if str(MIDI_MODEL_DIR) not in sys.path:
        sys.path.insert(0, str(MIDI_MODEL_DIR))

    try:
        from midi_model import MIDIModel, MIDIModelConfig
        from midi_tokenizer import MIDITokenizerV2
        import MIDI as MIDI_module

        # Charger la configuration
        print(f"Configuration : {config_name}")
        config = MIDIModelConfig.from_name(config_name)

        # Creer le modele
        model = MIDIModel(config=config)
        tokenizer = model.tokenizer

        # Telecharger et charger les poids
        print(f"Telechargement des poids depuis {model_id}...")
        from safetensors.torch import load_file

        weights_path = MIDI_MODEL_DIR / "model.safetensors"
        if not weights_path.exists():
            weights_path = hf_hub_download(
                repo_id=model_id,
                filename="model.safetensors",
                local_dir=str(MIDI_MODEL_DIR),
                token=hf_token
            )

        state_dict = load_file(str(weights_path))
        model.load_state_dict(state_dict, strict=False)

        # Choisir dtype et device
        dtype = torch.bfloat16 if device == "cuda" and gpu_available else torch.float32
        model = model.to(device, dtype=dtype).eval()

        midi_model_loaded = True
        param_count = sum(p.numel() for p in model.parameters()) / 1e6
        print(f"\nModele charge : {param_count:.0f}M parametres")
        print(f"Device : {device}, Dtype : {dtype}")
        print(f"Vocabulaire tokenizer : {tokenizer.vocab_size} tokens")
        print(f"Max token seq par evenement : {tokenizer.max_token_seq}")

        if gpu_available:
            vram_used = torch.cuda.memory_allocated(0) / (1024**3)
            print(f"VRAM utilisee : {vram_used:.2f} GB")

    except ImportError as e:
        print(f"Import manquant : {e}")
        print("Verifier que les fichiers source sont telecharges")
    except Exception as e:
        print(f"Erreur lors du chargement : {type(e).__name__} - {str(e)[:200]}")
else:
    print("Fichiers source non disponibles")

CHARGEMENT DU MODELE
Fichiers source non disponibles


### Interpretation : Chargement du modele

| Aspect | Valeur typique | Signification |
|--------|---------------|---------------|
| Parametres | ~100M (medium) | Beaucoup plus leger que MusicGen (300M-3.3B) |
| VRAM | ~0.5-2 GB | Accessible sur la plupart des GPU |
| Vocabulaire | 3406 tokens | Couvre tous les evenements MIDI standard |

Le modele medium offre un bon equilibre entre qualite et rapidite. La variante "o" (optimised) reorganise les pistes et canaux MIDI pour de meilleurs resultats.

## Section 2 : Generation inconditionnelle

La generation inconditionnelle laisse le modele composer librement sans contrainte. Le modele choisit le tempo, les instruments, la tonalite et le style.

| Parametre | Valeur | Role |
|-----------|--------|------|
| `max_len` | 512 | Nombre max d'evenements MIDI a generer |
| `temp` | 1.0 | Diversite du sampling |
| `top_k` | 20 | Filtre les k tokens les plus probables |
| `top_p` | 0.98 | Nucleus sampling |

In [6]:
# Fonctions utilitaires pour la generation et la synthese

def generate_and_save(model, tokenizer, prompt=None, max_len=512,
                      temp=1.0, top_k=20, top_p=0.98,
                      filename="output", output_dir=OUTPUT_DIR):
    """Genere du MIDI, sauvegarde le fichier .mid et retourne le score."""
    start_time = time.time()

    midi_seq = model.generate(
        prompt=prompt,
        batch_size=1,
        max_len=max_len,
        temp=temp,
        top_p=top_p,
        top_k=top_k
    )

    gen_time = time.time() - start_time
    num_events = midi_seq.shape[1]

    # Detokeniser en score MIDI
    mid_score = tokenizer.detokenize(midi_seq[0])

    # Sauvegarder en fichier .mid
    midi_bytes = MIDI_module.score2midi(mid_score)
    mid_path = output_dir / f"{filename}.mid"
    with open(str(mid_path), "wb") as f:
        f.write(midi_bytes)

    # Extraire les informations
    ticks_per_beat = mid_score[0] if mid_score else 480
    num_tracks = len(mid_score) - 1 if len(mid_score) > 1 else 0

    result = {
        "path": mid_path,
        "score": mid_score,
        "num_events": num_events,
        "num_tracks": num_tracks,
        "gen_time": gen_time,
        "ticks_per_beat": ticks_per_beat,
        "file_size": mid_path.stat().st_size
    }

    print(f"  Evenements : {num_events} | Pistes : {num_tracks}")
    print(f"  Temps : {gen_time:.2f}s | Fichier : {mid_path.name} ({result['file_size']} octets)")

    return result


def synthesize_midi(mid_score, soundfont_path, sample_rate=44100):
    """Synthetise un score MIDI en audio via FluidSynth."""
    if not fluidsynth_available or soundfont_path is None:
        print("  Synthese non disponible (FluidSynth ou soundfont manquant)")
        return None

    try:
        from midi_synthesizer import MidiSynthesizer
        synth = MidiSynthesizer(soundfont_path, sample_rate=sample_rate)
        mid_opus = MIDI_module.score2opus(mid_score)
        audio_data = synth.synthesis(mid_opus)
        return audio_data  # numpy int16 array, shape (samples, 2)
    except Exception as e:
        print(f"  Erreur synthese : {type(e).__name__} - {str(e)[:100]}")
        return None


def play_midi(mid_score, soundfont_path, sample_rate=44100):
    """Synthetise et joue le MIDI dans le notebook."""
    audio_data = synthesize_midi(mid_score, soundfont_path, sample_rate)
    if audio_data is not None:
        # Convertir int16 en float pour IPython.display.Audio
        audio_float = audio_data.astype(np.float32) / 32768.0
        display(Audio(data=audio_float.T, rate=sample_rate))
        return audio_data
    return None


print("Fonctions utilitaires definies")

Fonctions utilitaires definies


Les fonctions de generation MIDI et de synthese audio FluidSynth sont definies. La cellule suivante lance la generation inconditionnelle : le modele compose 3 morceaux sans contrainte de style ou d'instrument.

In [7]:
# Generation inconditionnelle
print("GENERATION INCONDITIONNELLE")
print("=" * 45)

uncond_results = {}

if midi_model_loaded and generate_midi:
    for i in range(3):
        print(f"\n--- Morceau {i+1} ---")

        result = generate_and_save(
            model, tokenizer,
            prompt=None,
            max_len=max_len,
            temp=temperature,
            top_k=top_k,
            top_p=top_p,
            filename=f"uncond_{i+1}"
        )
        uncond_results[f"morceau_{i+1}"] = result

        # Synthese audio
        if synthesize_audio:
            audio = play_midi(result["score"], soundfont_path)
            if audio is not None and save_results:
                import scipy.io.wavfile
                wav_path = OUTPUT_DIR / f"uncond_{i+1}.wav"
                scipy.io.wavfile.write(str(wav_path), 44100, audio)
                print(f"  Audio sauvegarde : {wav_path.name}")

    # Recapitulatif
    print(f"\nRecapitulatif :")
    print(f"{'Morceau':<12} {'Events':<10} {'Pistes':<10} {'Temps (s)':<12} {'Taille (B)':<12}")
    print("-" * 56)
    for name, data in uncond_results.items():
        print(f"{name:<12} {data['num_events']:<10} {data['num_tracks']:<10} "
              f"{data['gen_time']:<12.2f} {data['file_size']:<12}")
else:
    print("Modele non charge ou generation desactivee")

GENERATION INCONDITIONNELLE
Modele non charge ou generation desactivee


### Interpretation : Generation inconditionnelle

| Aspect | Valeur typique | Signification |
|--------|---------------|---------------|
| Evenements | 200-512 | Nombre de notes/changements generes |
| Pistes | 2-8 | Le modele attribue automatiquement les instruments |
| Temps | 5-30s (GPU) | Beaucoup plus rapide que MusicGen |
| Taille .mid | 1-10 KB | Extremement compact vs audio |

**Points cles** :
1. Chaque generation est unique et non-deterministe
2. Le modele choisit librement les instruments, le tempo et la structure
3. Les fichiers MIDI sont editables dans n'importe quel DAW (FL Studio, Ableton, MuseScore)

## Section 3 : Prompts musicaux

Contrairement a MusicGen qui utilise des descriptions textuelles, midi-model accepte des **prompts symboliques** : des sequences d'evenements MIDI qui definissent le contexte musical.

On peut specifier :

| Element | Methode | Exemple |
|---------|---------|--------|
| Tempo | `set_tempo` | 120 BPM |
| Signature | `time_signature` | 4/4, 3/4, 6/8 |
| Tonalite | `key_signature` | Do majeur, La mineur |
| Instruments | `patch_change` | Piano (0), Guitare (25), Violon (40) |

In [8]:
# Prompts musicaux - Configuration d'instruments et de tempo
print("GENERATION AVEC PROMPTS MUSICAUX")
print("=" * 45)

# Instruments MIDI General (numeros de patch)
INSTRUMENTS = {
    "Acoustic Grand Piano": 0,
    "Electric Piano": 4,
    "Nylon Guitar": 24,
    "Electric Guitar": 27,
    "Acoustic Bass": 32,
    "Electric Bass": 33,
    "Violin": 40,
    "Cello": 42,
    "String Ensemble": 48,
    "Trumpet": 56,
    "Alto Sax": 65,
    "Flute": 73,
    "Synth Lead": 80,
    "Synth Pad": 88,
}


def create_music_prompt(tokenizer, tempo=120, time_sig=(4, 4),
                        key=("C", "major"), instruments=None):
    """Cree un prompt MIDI avec tempo, signature et instruments."""
    mid = [[tokenizer.bos_id] + [tokenizer.pad_id] * (tokenizer.max_token_seq - 1)]

    # Tempo
    mid.append(tokenizer.event2tokens(["set_tempo", 0, 0, 0, tempo]))

    # Signature temporelle : nn = numerator-1, dd = denominator_power-1
    nn = time_sig[0] - 1
    dd = {1: 0, 2: 0, 4: 1, 8: 2, 16: 3}.get(time_sig[1], 1)
    mid.append(tokenizer.event2tokens(["time_signature", 0, 0, 0, nn, dd]))

    # Tonalite : sf+7 (sf va de -7 a +7), mi (0=majeur, 1=mineur)
    key_map = {"C": 0, "G": 1, "D": 2, "A": 3, "E": 4, "B": 5,
               "F#": 6, "Gb": -6, "F": -1, "Bb": -2, "Eb": -3,
               "Ab": -4, "Db": -5, "Cb": -7}
    sf = key_map.get(key[0], 0) + 7
    mi = 0 if key[1] == "major" else 1
    mid.append(tokenizer.event2tokens(["key_signature", 0, 0, 0, sf, mi]))

    # Instruments
    if instruments:
        for track_idx, (channel, patch) in enumerate(instruments, start=1):
            mid.append(tokenizer.event2tokens(
                ["patch_change", 0, 0, track_idx, channel, patch]
            ))

    return np.array([mid], dtype=np.int64)


# Definir des presets musicaux
presets = {
    "Piano classique": {
        "tempo": 100,
        "time_sig": (4, 4),
        "key": ("C", "major"),
        "instruments": [(0, 0)]  # Piano sur canal 0
    },
    "Jazz quartet": {
        "tempo": 130,
        "time_sig": (4, 4),
        "key": ("Bb", "major"),
        "instruments": [(0, 0), (1, 32), (2, 65), (3, 0)]  # Piano, Bass, Sax, Piano
    },
    "Valse romantique": {
        "tempo": 160,
        "time_sig": (3, 4),
        "key": ("D", "major"),
        "instruments": [(0, 0), (1, 40), (2, 42)]  # Piano, Violon, Cello
    },
    "Synth ambient": {
        "tempo": 80,
        "time_sig": (4, 4),
        "key": ("A", "minor"),
        "instruments": [(0, 88), (1, 80), (2, 48)]  # Pad, Lead, Strings
    },
}

prompt_results = {}

if midi_model_loaded and generate_midi and test_prompts:
    for name, params in presets.items():
        print(f"\n--- {name} ---")
        print(f"  Tempo: {params['tempo']} BPM | Signature: {params['time_sig'][0]}/{params['time_sig'][1]}")
        print(f"  Tonalite: {params['key'][0]} {params['key'][1]}")

        prompt = create_music_prompt(
            tokenizer,
            tempo=params["tempo"],
            time_sig=params["time_sig"],
            key=params["key"],
            instruments=params["instruments"]
        )

        safe_name = name.lower().replace(" ", "_")
        result = generate_and_save(
            model, tokenizer,
            prompt=prompt,
            max_len=max_len,
            temp=temperature,
            top_k=top_k,
            top_p=top_p,
            filename=f"prompt_{safe_name}"
        )
        prompt_results[name] = result

        if synthesize_audio:
            audio = play_midi(result["score"], soundfont_path)
            if audio is not None and save_results:
                import scipy.io.wavfile
                wav_path = OUTPUT_DIR / f"prompt_{safe_name}.wav"
                scipy.io.wavfile.write(str(wav_path), 44100, audio)

    # Recapitulatif
    print(f"\nRecapitulatif des presets :")
    print(f"{'Preset':<22} {'Events':<10} {'Pistes':<10} {'Temps (s)':<12}")
    print("-" * 54)
    for name, data in prompt_results.items():
        print(f"{name:<22} {data['num_events']:<10} {data['num_tracks']:<10} "
              f"{data['gen_time']:<12.2f}")
else:
    print("Modele non charge, generation desactivee, ou prompts desactives")

GENERATION AVEC PROMPTS MUSICAUX
Modele non charge, generation desactivee, ou prompts desactives


### Interpretation : Prompts musicaux

| Preset | Observation attendue |
|--------|---------------------|
| Piano classique | Piece solo, tonalite claire |
| Jazz quartet | Polyphonie riche, rythme swing |
| Valse romantique | Rythme ternaire 3/4, expressif |
| Synth ambient | Textures longues, atmospherique |

**Points cles** :
1. Le prompt definit le **contexte initial**, pas le contenu exact
2. Le modele peut ajouter des instruments supplementaires au-dela du prompt
3. Le tempo et la tonalite sont generalement respectes
4. La signature temporelle influence fortement le ressenti rhythmique

## Section 4 : Parametres de sampling

Comme pour les LLM textuels, les parametres de sampling controlent la diversite et la qualite de la generation.

| Parametre | Valeur par defaut | Plage | Impact |
|-----------|-------------------|-------|--------|
| `temp` | 1.0 | 0.1-1.2 | Diversite globale |
| `top_k` | 20 | 1-128 | Filtre les tokens improbables |
| `top_p` | 0.98 | 0.1-1.0 | Nucleus sampling |

### Presets recommandes

| Style | temp | top_k | top_p | Resultat |
|-------|------|-------|-------|----------|
| Conservateur | 1.0 | 128 | 0.94 | Previsible, classique |
| Equilibre | 1.0 | 20 | 0.98 | Bon compromis |
| Creatif | 1.0 | 12 | 0.98 | Surprenant, varie |

In [9]:
# Test des parametres de sampling
print("TEST DES PARAMETRES DE SAMPLING")
print("=" * 45)

sampling_configs = {
    "Conservateur": {"temp": 0.8, "top_k": 128, "top_p": 0.94},
    "Equilibre":    {"temp": 1.0, "top_k": 20,  "top_p": 0.98},
    "Creatif":      {"temp": 1.1, "top_k": 12,  "top_p": 0.98},
}

sampling_results = {}

if midi_model_loaded and generate_midi:
    # Prompt commun : piano solo
    common_prompt = create_music_prompt(
        tokenizer, tempo=110, time_sig=(4, 4),
        key=("C", "major"), instruments=[(0, 0)]
    )

    for name, params in sampling_configs.items():
        print(f"\n--- {name} (temp={params['temp']}, top_k={params['top_k']}, top_p={params['top_p']}) ---")

        safe_name = name.lower()
        result = generate_and_save(
            model, tokenizer,
            prompt=common_prompt,
            max_len=max_len,
            temp=params["temp"],
            top_k=params["top_k"],
            top_p=params["top_p"],
            filename=f"sampling_{safe_name}"
        )
        sampling_results[name] = result

        if synthesize_audio:
            play_midi(result["score"], soundfont_path)

    # Recapitulatif
    print(f"\nComparaison :")
    print(f"{'Config':<16} {'temp':<8} {'top_k':<8} {'top_p':<8} {'Events':<10} {'Pistes':<10}")
    print("-" * 60)
    for name, data in sampling_results.items():
        cfg = sampling_configs[name]
        print(f"{name:<16} {cfg['temp']:<8} {cfg['top_k']:<8} {cfg['top_p']:<8} "
              f"{data['num_events']:<10} {data['num_tracks']:<10}")
else:
    print("Modele non charge ou generation desactivee")

TEST DES PARAMETRES DE SAMPLING
Modele non charge ou generation desactivee


### Interpretation : Parametres de sampling

| Configuration | Observation |
|--------------|-------------|
| Conservateur | Melodie previsible, accords standards, peu de surprises |
| Equilibre | Bon compromis entre coherence et originalite |
| Creatif | Harmonies inattendues, rythmes varies, parfois incoherent |

**Points cles** :
1. Contrairement aux LLM textuels, la temperature n'a pas besoin d'etre tres basse pour des resultats coherents
2. Le `top_k` a plus d'impact que la temperature sur la diversite instrumentale
3. Un `top_p` proche de 1.0 permet plus de diversite harmonique

## Section 5 : Continuation de MIDI existant

midi-model peut continuer une sequence MIDI existante. On tokenise un fichier MIDI, on en prend les premiers evenements comme prompt, et le modele genere la suite.

| Etape | Description |
|-------|-------------|
| 1. Charger | Lire le fichier .mid |
| 2. Tokeniser | Convertir en tokens via MIDITokenizerV2 |
| 3. Tronquer | Garder les N premiers evenements comme contexte |
| 4. Generer | Le modele complete la sequence |

In [10]:
# Continuation d'un MIDI genere precedemment
print("CONTINUATION MIDI")
print("=" * 45)

if midi_model_loaded and generate_midi and uncond_results:
    # Utiliser le premier morceau genere comme base
    source = uncond_results.get("morceau_1")
    if source and source["path"].exists():
        print(f"Source : {source['path'].name}")

        # Lire le fichier MIDI
        with open(str(source["path"]), "rb") as f:
            midi_data = f.read()

        # Tokeniser
        midi_score = MIDI_module.midi2score(midi_data)
        tokens = tokenizer.tokenize(
            midi_score,
            cc_eps=4,
            tempo_eps=4,
            remap_track_channel=True,
            add_default_instr=True,
            remove_empty_channels=False
        )

        # Prendre les 128 premiers evenements comme prompt
        prompt_len = min(128, len(tokens))
        prompt = np.array([tokens[:prompt_len]], dtype=np.int64)
        print(f"Prompt : {prompt_len} evenements (sur {len(tokens)} total)")

        # Generer la suite
        print(f"\nGeneration de la continuation...")
        result = generate_and_save(
            model, tokenizer,
            prompt=prompt,
            max_len=max_len,
            temp=temperature,
            top_k=top_k,
            top_p=top_p,
            filename="continuation"
        )

        if synthesize_audio:
            print("\nOriginal :")
            play_midi(source["score"], soundfont_path)
            print("\nContinuation :")
            play_midi(result["score"], soundfont_path)
    else:
        print("Pas de morceau source disponible")
else:
    print("Modele non charge, generation desactivee, ou pas de morceaux precedents")

CONTINUATION MIDI
Modele non charge, generation desactivee, ou pas de morceaux precedents


### Interpretation : Continuation

| Aspect | Observation |
|--------|-------------|
| Coherence stylistique | La continuation conserve le style du morceau source |
| Instruments | Les memes instruments sont generalement maintenus |
| Transitions | Parfois abruptes, surtout avec peu de contexte |

**Points cles** :
1. Plus le prompt est long, plus la continuation est coherente
2. La fenetre de contexte est limitee a 4096 evenements
3. On peut utiliser un fichier MIDI externe (composition personnelle, partition)

## Section 6 : Comparaison MIDI vs Audio (MusicGen)

Recapitulatif comparatif entre l'approche symbolique (ce notebook) et l'approche audio directe (notebook 02-3).

| Critere | midi-model (MIDI) | MusicGen (Audio) |
|---------|-------------------|------------------|
| **Entree** | Prompt symbolique (instruments, tempo) | Description textuelle naturelle |
| **Sortie** | Fichier .mid (notes, durees) | Fichier .wav (signal audio) |
| **Editabilite** | Totale (note par note) | Aucune |
| **Multi-instruments** | Natif (16 canaux MIDI) | Melange dans le signal |
| **Qualite sonore** | Depend du synthetiseur (SoundFont) | Directement realiste |
| **VRAM** | ~2-4 GB | ~10-20 GB |
| **Taille sortie** | ~1-10 KB (.mid) | ~1-5 MB (.wav pour 10s) |
| **Cas d'usage** | Composition, arrangement, education | Musique de fond, ambiance |
| **Post-production** | DAW (FL Studio, Ableton, MuseScore) | Edition audio limitee |

### Quand utiliser chaque approche ?

| Besoin | Recommandation |
|--------|----------------|
| Musique de fond pour video | MusicGen (resultat sonore direct) |
| Composition assistee | midi-model (edition note par note) |
| Prototypage rapide | midi-model (leger, rapide) |
| Qualite production | MusicGen + post-production audio |
| Education musicale | midi-model (visualisation des notes) |
| GPU limite | midi-model (~2 GB vs ~10 GB) |

In [11]:
# Mode interactif - Generation personnalisee
if notebook_mode == "interactive" and not skip_widgets:
    print("MODE INTERACTIF - GENERATION MIDI PERSONNALISEE")
    print("=" * 55)

    print("\nChoisissez un preset ou creez votre propre configuration :")
    print("  1. Piano solo (tempo libre)")
    print("  2. Duo violon + piano")
    print("  3. Orchestre (cordes + bois + cuivres)")
    print("  4. Generation libre (inconditionnelle)")
    print("  (Laissez vide pour passer)")

    try:
        choice = input("\nChoix [1-4] : ").strip()

        if choice and midi_model_loaded:
            user_presets = {
                "1": {"name": "Piano solo", "tempo": 100, "instruments": [(0, 0)]},
                "2": {"name": "Duo violon-piano", "tempo": 90, "instruments": [(0, 0), (1, 40)]},
                "3": {"name": "Orchestre", "tempo": 80, "instruments": [(0, 48), (1, 40), (2, 73), (3, 56)]},
            }

            if choice in user_presets:
                p = user_presets[choice]
                print(f"\nGeneration : {p['name']}")
                prompt = create_music_prompt(
                    tokenizer, tempo=p["tempo"],
                    instruments=p["instruments"]
                )
                result = generate_and_save(
                    model, tokenizer, prompt=prompt,
                    max_len=max_len, temp=temperature,
                    top_k=top_k, top_p=top_p,
                    filename=f"custom_{p['name'].lower().replace(' ', '_')}"
                )
                if synthesize_audio:
                    play_midi(result["score"], soundfont_path)
            elif choice == "4":
                print("\nGeneration libre...")
                result = generate_and_save(
                    model, tokenizer, prompt=None,
                    max_len=max_len, temp=temperature,
                    top_k=top_k, top_p=top_p,
                    filename="custom_libre"
                )
                if synthesize_audio:
                    play_midi(result["score"], soundfont_path)
            else:
                print("Choix non reconnu")
        else:
            print("Mode interactif ignore")

    except (KeyboardInterrupt, EOFError):
        print("Mode interactif interrompu")
    except Exception as e:
        error_type = type(e).__name__
        if "StdinNotImplemented" in error_type or "input" in str(e).lower():
            print("Mode interactif non disponible (execution automatisee)")
        else:
            print(f"Erreur : {error_type} - {str(e)[:100]}")
else:
    print("Mode batch - Interface interactive desactivee")

MODE INTERACTIF - GENERATION MIDI PERSONNALISEE

Choisissez un preset ou creez votre propre configuration :
  1. Piano solo (tempo libre)
  2. Duo violon + piano
  3. Orchestre (cordes + bois + cuivres)
  4. Generation libre (inconditionnelle)
  (Laissez vide pour passer)
Mode interactif non disponible (execution automatisee)


## Bonnes pratiques et guide de generation

### Construire de bons prompts

| Element | Bon exemple | Mauvais exemple |
|---------|------------|----------------|
| Instruments | 2-4 instruments complementaires | 10 instruments simultanes |
| Tempo | Adapte au genre (jazz ~130, ambient ~80) | Tempo extreme (>200 ou <40) |
| Tonalite | Coherente avec le genre | Tonalite aleatoire |

### Limites actuelles

| Limite | Description | Contournement |
|--------|-------------|---------------|
| Pas de texte | Prompt symbolique, pas de description textuelle | Utiliser MusicGen pour text-to-music |
| Duree variable | Le nombre d'evenements ne correspond pas a une duree fixe | Ajuster `max_len` iterativement |
| Qualite son | Depend du SoundFont utilise | Utiliser des SoundFont de haute qualite |
| Pas de paroles | MIDI est instrumental | Combiner avec un modele vocal |

In [12]:
# Statistiques de session
print("STATISTIQUES DE SESSION")
print("=" * 45)

print(f"Date : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Modele : {model_id} ({config_name})")
print(f"Device : {device}")
print(f"Max events : {max_len}")
print(f"Parametres : temp={temperature}, top_k={top_k}, top_p={top_p}")
print(f"Modele charge : {'Oui' if midi_model_loaded else 'Non'}")
print(f"FluidSynth : {'Oui' if fluidsynth_available else 'Non'}")

if gpu_available:
    vram_current = torch.cuda.memory_allocated(0) / (1024**3)
    print(f"VRAM utilisee : {vram_current:.2f} GB")

if save_results:
    mid_files = list(OUTPUT_DIR.glob('*.mid'))
    wav_files = list(OUTPUT_DIR.glob('*.wav'))
    mid_size = sum(f.stat().st_size for f in mid_files)
    wav_size = sum(f.stat().st_size for f in wav_files) / (1024*1024)
    print(f"\nFichiers MIDI : {len(mid_files)} ({mid_size} octets)")
    print(f"Fichiers WAV : {len(wav_files)} ({wav_size:.1f} MB)")
    print(f"Repertoire : {OUTPUT_DIR}")

# Liberation memoire
if midi_model_loaded:
    print(f"\nLiberation du modele...")
    del model
    gc.collect()
    if gpu_available:
        torch.cuda.empty_cache()
    print(f"Memoire liberee")

print(f"\nPROCHAINES ETAPES")
print(f"1. Comparer avec MusicGen (02-3) pour choisir l'approche adaptee")
print(f"2. Editer les fichiers .mid dans un DAW (MuseScore, FL Studio)")
print(f"3. Explorer les LoRA J-Pop et Touhou pour des styles specifiques")
print(f"4. Combiner MIDI + TTS pour des compositions avec voix")

print(f"\nNotebook Generation MIDI termine - {datetime.now().strftime('%H:%M:%S')}")

STATISTIQUES DE SESSION
Date : 2026-03-22 13:34:30
Modele : skytnt/midi-model-tv2o-medium (tv2o-medium)
Device : cpu
Max events : 512
Parametres : temp=1.0, top_k=20, top_p=0.98
Modele charge : Non
FluidSynth : Non

Fichiers MIDI : 0 (0 octets)
Fichiers WAV : 0 (0.0 MB)
Repertoire : D:\Dev\CoursIA.worktrees\GenAI_Series\MyIA.AI.Notebooks\GenAI\outputs\audio\midi

PROCHAINES ETAPES
1. Comparer avec MusicGen (02-3) pour choisir l'approche adaptee
2. Editer les fichiers .mid dans un DAW (MuseScore, FL Studio)
3. Explorer les LoRA J-Pop et Touhou pour des styles specifiques
4. Combiner MIDI + TTS pour des compositions avec voix

Notebook Generation MIDI termine - 13:34:30


---

# EXERCICE - Composition guidee multi-instruments

## Objectif

Composer une piece MIDI multi-instruments en utilisant les prompts musicaux et comparer differents parametres de sampling.

## Consignes

1. **Choisir un contexte musical** parmi :
   - Bande-son de jeu video (RPG medieval)
   - Musique d'ascenseur (jazz leger)
   - Generique de podcast (moderne, court)

2. **Configurer le prompt** :
   - Choisir 3-4 instruments adaptes au contexte
   - Definir le tempo, la tonalite et la signature temporelle
   - Justifier vos choix musicaux

3. **Generer 3 variations** :
   - Variation A : parametres conservateurs (top_k=128, temp=0.8)
   - Variation B : parametres equilibres (top_k=20, temp=1.0)
   - Variation C : parametres creatifs (top_k=12, temp=1.1)

4. **Analyser et comparer** :
   - Ecouter les 3 variations synthetisees
   - Noter chaque variation sur : coherence (1-5), originalite (1-5), adequation au contexte (1-5)
   - Identifier la meilleure variation et expliquer pourquoi

## Indices

- Utilisez `create_music_prompt()` pour configurer le prompt
- Utilisez `generate_and_save()` avec differents parametres
- Utilisez `play_midi()` pour ecouter les resultats
- Consultez le dictionnaire `INSTRUMENTS` pour les numeros de patch MIDI

## Criteres de succes

- [ ] Prompt musical coherent avec le contexte choisi
- [ ] 3 variations generees avec parametres differents
- [ ] Tableau comparatif avec notes et justification
- [ ] Fichiers .mid et .wav sauvegardes

---

**Soumission** : PR avec titre "Exercice MIDI - [Votre Nom]", fichiers generes et tableau comparatif